In [0]:
#window Function

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
# read the data from csv
path="/Volumes/workspace/default/csv_data/tallest_people_in_the_world.csv"
df=spark.read.format("csv").option("header","true").option("inferSchema","true").load(path)
df.display()


In [0]:
# find 2nd Tallest person in the world
window_spec = Window.partitionBy("country").orderBy(F.col("height_cm").desc())

new_df=(
    df
    .withColumn("Rank", F.dense_rank().over(window_spec))
    .filter(F.col("Rank") == 2)
    .drop("Rank")
)

new_df.display()

In [0]:
window_spec = Window.partitionBy("country").orderBy(F.col("height_cm").desc())

new_df=(
    df
    .withColumn("Rank", F.rank().over(window_spec))
    .filter(F.col("Rank") == 2)
    .drop("Rank")
)

new_df.display()

In [0]:
#Joins

In [0]:
# Read tyhe table from the Sample database

cus_df=spark.table("samples.tpch.customer")
cus_df.display()

In [0]:
order_df=spark.table("samples.tpch.orders")
order_df.display()

In [0]:
# Find the customer placesed most number of orders

Prim_cus_df = (
    cus_df.select(
        F.col("c_custkey").alias("customerID"),
        F.col("c_name").alias("customer_name")    
    )
    .join(
        order_df.select(
            F.col("o_custkey").alias('customerID'),
            F.col("o_orderkey").alias('orderID')
           ),
           on =[ "customerID"],
            how = "inner"
        
    )
)
Prim_cus_df.display()

In [0]:
val_cus_df=(
Prim_cus_df
.groupBy("customerID","customer_name")
.agg(F.count("orderID").alias("order_count"))
.orderBy(F.col("order_count").desc())
.limit(1)
)
val_cus_df.display()

In [0]:
# find the coutomer who have not ordered anything yet

non_cus_df = (
    cus_df.select(
        F.col("c_custkey").alias("customerID"),
        F.col("c_name").alias("customer_name")    
    )
    .join(
        order_df.select(
            F.col("o_custkey").alias('customerID'),
            F.col("o_orderkey").alias('orderID')
           ),
           on =[ "customerID"],
            how = "leftanti"
        
    )
)
non_cus_df.display()
